# Semantic Kernel 

In this code sample, you will use the [Semantic Kernel](https://aka.ms/ai-agents-beginners/semantic-kernel) AI Framework to create a basic agent. 

The goal of this sample is to show you the steps that we will later use in the additional code samples when implementing the different agentic patterns. 

## Import the Needed Python Packages 

In [30]:
import os 
from typing import Annotated
from openai import AsyncOpenAI

from dotenv import load_dotenv

from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.functions import kernel_function

## Creating the Client

In this sample, we will use [GitHub Models](https://aka.ms/ai-agents-beginners/github-models) for access to the LLM. 

The `ai_model_id` is defined as `gpt-4o-mini`. Try changing the model to another model available on the GitHub Models marketplace to see the different results. 

For us to use the `Azure Inference SDK` that is used for the `base_url` for GitHub Models, we will use the `OpenAIChatCompletion` connector within Semantic Kernel. There are also other [available connectors](https://learn.microsoft.com/semantic-kernel/concepts/ai-services/chat-completion) to use Semantic Kernel for other model providers.

In [31]:
import random   

# Define a sample plugin for the sample

class DestinationsPlugin:
    """A List of Random Destinations for a vacation."""

    def __init__(self):
        # List of vacation destinations
        self.destinations = [
            "Barcelona, Spain",
            "Paris, France",
            "Berlin, Germany",
            "Tokyo, Japan",
            "Sydney, Australia",
            "New York, USA",
            "Cairo, Egypt",
            "Cape Town, South Africa",
            "Rio de Janeiro, Brazil",
            "Bali, Indonesia"
        ]
        # Track last destination to avoid repeats
        self.last_destination = None

    @kernel_function(description="Provides a random vacation destination.")
    def get_random_destination(self) -> Annotated[str, "Returns a random vacation destination."]:
        # Get available destinations (excluding last one if possible)
        available_destinations = self.destinations.copy()
        if self.last_destination and len(available_destinations) > 1:
            available_destinations.remove(self.last_destination)

        # Select a random destination
        destination = random.choice(available_destinations)

        # Update the last destination
        self.last_destination = destination

        return destination

In [ ]:
load_dotenv("/Users/mikeqin/code/ml_learning/ai-agents-for-beginners/.env_github", override=True)
client = AsyncOpenAI(
    api_key=os.environ.get("GITHUB_TOKEN"), 
    base_url="https://models.inference.ai.azure.com/",
)

# Create an AI Service that will be used by the `ChatCompletionAgent`
chat_completion_service = OpenAIChatCompletion(
    ai_model_id="gpt-4o-mini",
    async_client=client,
)

print("pause here")

In [33]:
client


## Creating the Agent 

Below we create the Agent called `TravelAgent`.

For this example, we are using very simple instructions. You can change these instructions to see how the agent responds differently. 

In [35]:
agent = ChatCompletionAgent(
    service=chat_completion_service, 
    plugins=[DestinationsPlugin()],
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
)

In [36]:
agent

ChatCompletionAgent(arguments=None, description=None, id='54b95c9c-03ae-4c53-8708-8716f3b5764a', instructions='You are a helpful AI Agent that can help plan vacations for customers at random destinations', kernel=Kernel(retry_mechanism=PassThroughWithoutRetry(), services={'gpt-4o-mini': OpenAIChatCompletion(ai_model_id='gpt-4o-mini', service_id='gpt-4o-mini', instruction_role='system', client=<openai.AsyncOpenAI object at 0x31e807fb0>, ai_model_type=<OpenAIModelTypes.CHAT: 'chat'>, prompt_tokens=0, completion_tokens=0, total_tokens=0)}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AIServiceSelector object at 0x31eb2c530>, plugins={'DestinationsPlugin': KernelPlugin(name='DestinationsPlugin', description=None, functions={'get_random_destination': KernelFunctionFromMethod(metadata=KernelFunctionMetadata(name='get_random_destination', plugin_name='DestinationsPlugin', description='Provides a random vacation destination.', parameters=[], is_prompt=False, is_asynchronou

## Running the Agent

Now we can run the Agent by defining a thread of type `ChatHistoryAgentThread`.  Any required system messages are provided to the agent's invoke_stream `messages` keyword argument.

After these are defined, we create a `user_inputs` that will be what the user is sending to the agent. In this case, we have set this message to `Plan me a sunny vacation`. 

Feel free to change this message to see how the agent responds differently. 

In [37]:

async def main():

    # Create a new thread for the agent
    # If no thread is provided, a new thread will be
    # created and returned with the initial response
    thread: ChatHistoryAgentThread | None = None

    user_inputs = [
        "Plan me a day trip.",
    ]

    for user_input in user_inputs:

        print(f"# User: {user_input}\n")
        first_chunk = True
        i = 0
        async for response in agent.invoke_stream(
            messages=user_input, thread=thread,
        ):
            # 5. Print the response
            if first_chunk:
                print(f"# {response.name} ---- ", end="", flush=True)
                first_chunk = False
            print(f"{response}", end="", flush=True)
            thread = response.thread
            i += 1
        print(f"i={i}")

    # Clean up the thread
    await thread.delete() if thread else None

await main()

# User: Plan me a day trip.

# TravelAgent ---- How about a day trip to Rio de Janeiro, Brazil? Here’s a suggested itinerary:

### Morning
- **Breakfast at Confeitaria Colombo**: Start your day with a traditional Brazilian breakfast at this historic café known for its beautiful decor and delicious pastries.
- **Visit Christ the Redeemer**: Head to the iconic Christ the Redeemer statue early in the morning to avoid crowds. Enjoy breathtaking views of the city from the top.

### Afternoon
- **Lunch at a Local Churrascaria**: Experience a traditional Brazilian barbecue at a local churrascaria, where you can try various cuts of meat served at your table.
- **Explore Sugarloaf Mountain**: Take the cable car up to Sugarloaf Mountain for stunning panoramic views of Rio and the surrounding beaches.

### Evening
- **Relax at Copacabana Beach**: Spend some time lounging on the famous Copacabana Beach. You can enjoy a refreshing drink from one of the beachside kiosks.
- **Dinner at a Beachfront R